In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from pathlib import Path
import os

In [13]:
data_pipeline = "no-feature-eng"
dataset_type = "features"

In [14]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [15]:
raw_train_id = pd.read_csv(f"{data_path}/raw/train.csv")["id"]
X = pd.read_csv(Path(data_path) / Path(data_pipeline) / f"train_{dataset_type}.csv")
X_test = pd.read_csv(Path(data_path) / Path(data_pipeline) / f"test_{dataset_type}.csv")
y = pd.read_csv(Path(data_path) / Path(data_pipeline) / "train_labels.csv")

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   age                      662440 non-null  float64
 1   daily_screen_time_hours  595515 non-null  float64
 2   social_media_hours       557374 non-null  float64
 3   gaming_hours             564548 non-null  float64
 4   work_study_hours         639851 non-null  float64
 5   sleep_hours              646889 non-null  float64
 6   notifications_per_day    623785 non-null  float64
 7   app_opens_per_day        610659 non-null  float64
 8   weekend_screen_time      579306 non-null  float64
 9   gender                   662335 non-null  str    
 10  stress_level             636221 non-null  float64
 11  academic_work_impact     647145 non-null  float64
dtypes: float64(11), str(1)
memory usage: 63.3 MB


In [16]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [17]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns

for frame in [X, X_test]:
    for col in cat_cols:
        frame[col] = frame[col].astype('category')

In [20]:
kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)
TE_cols = [f"{col}_TE" for col in X.columns]
X_te = pd.DataFrame(index=X.index, columns=TE_cols, dtype=float)

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    te = TargetEncoder(smooth="auto")
    te.fit(X_train, y_train)
    X_te.iloc[valid_index] = te.transform(X_valid)

te = TargetEncoder(smooth="auto")
te.fit(X, y)
X_test_te = pd.DataFrame(
    te.transform(X_test),
    columns=TE_cols,
    index=X_test.index
)
X_te

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning

,age_TE,daily_screen_time_hours_TE,social_media_hours_TE,gaming_hours_TE,work_study_hours_TE,sleep_hours_TE,notifications_per_day_TE,app_opens_per_day_TE,weekend_screen_time_TE,gender_TE,stress_level_TE,academic_work_impact_TE
0,0.774607,0.711875,0.677908,0.652867,0.741855,0.750312,0.546153,0.951509,0.707474,0.722897,0.706498,0.710566
1,0.720036,0.530394,0.275946,0.710637,0.735232,0.743150,0.665984,0.377499,0.710417,0.703346,0.704888,0.711366
2,0.656416,0.399903,0.709732,0.710082,0.710639,0.722063,0.618738,0.614381,0.311960,0.704542,0.711187,0.708067
3,0.698706,0.358647,0.463453,0.632444,0.870486,0.804242,0.626155,0.678335,0.686288,0.700935,0.711004,0.709253
4,0.777572,1.000000,0.643466,0.934958,0.656844,0.626195,0.709986,0.712309,1.000000,0.704093,0.706498,0.710566
...,...,...,...,...,...,...,...,...,...,...,...,...
691364,0.741288,0.997978,0.901295,0.712424,0.678792,0.541565,0.909429,0.674724,1.000000,0.722661,0.708740,0.710849
691365,0.693304,0.274878,0.740501,0.525985,0.451237,0.773104,0.937756,0.884282,0.662880,0.703346,0.712148,0.711366
691366,0.711590,0.711875,0.709195,0.710372,0.642235,0.688496,0.709986,0.799768,0.378828,0.701156,0.706498,0.710566
691367,0.740969,1.000000,0.973571,0.571571,0.693705,0.676576,0.840842,0.637741,0.414640,0.700935,0.709602,0.709253


In [21]:
X_te.to_csv(Path(data_path) / Path(data_pipeline) / f"train_{dataset_type}_TE.csv", index=False)
X_test_te.to_csv(Path(data_path) / Path(data_pipeline) / f"test_{dataset_type}_TE.csv", index=False)